# MSTR T029 — Ministral Q4 Recovery (Colab)

Canonical fallback execution surface for the remaining `ministral-3-3b` T029 cell.

- executor: ephemeral Google Colab VM
- model source: public Hugging Face HTTPS GET only; no provider authentication or gated terms
- monetary ceiling: USD 0.00
- exact T027 candidate revision/integrity envelope remains authoritative
- exact llama.cpp commit: `fc35562ba46fbbf8e30cac85edbb39642c37d248`
- binaries stay under `/content` and are deleted by the runner; only the JSON report is durable
- no inference, training, release, or B011 candidate access is authorized here

Prepare `/content/mstr` as a checkout of PR #95 (or another checkout containing the exact runner blob below), then run all cells.


In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/content/mstr')
RUNNER = REPO / 'colab/mstr_t029_quantize.py'
MANIFEST = REPO / 'artifacts/manifests/T027-weight-access.json'
OUT = Path('/content/reports')
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_RUNNER_GIT_BLOB = '63c81229d0c797ea0347255f0916d0b7ed9a9514'
EXPECTED_MANIFEST_GIT_BLOB = 'ef73095e2e9c5bdcca7147d4bdeb92a5aa9a6d0f'
LLAMA_CPP_COMMIT = 'fc35562ba46fbbf8e30cac85edbb39642c37d248'
CANDIDATE = 'ministral-3-3b'

assert RUNNER.is_file(), RUNNER
assert MANIFEST.is_file(), MANIFEST
runner_blob = subprocess.check_output(['git','-C',str(REPO),'hash-object',str(RUNNER)], text=True).strip()
manifest_blob = subprocess.check_output(['git','-C',str(REPO),'hash-object',str(MANIFEST)], text=True).strip()
assert runner_blob == EXPECTED_RUNNER_GIT_BLOB, (runner_blob, EXPECTED_RUNNER_GIT_BLOB)
assert manifest_blob == EXPECTED_MANIFEST_GIT_BLOB, (manifest_blob, EXPECTED_MANIFEST_GIT_BLOB)
print('Pinned repository inputs verified.')


In [ ]:
!python -m pip install -q numpy sentencepiece transformers safetensors protobuf
!cmake --version
!git --version


In [ ]:
import json
manifest = json.loads(MANIFEST.read_text(encoding='utf-8'))
candidate = next(c for c in manifest['candidates'] if c['candidate_id'] == CANDIDATE)
assert candidate['rights_decision'] == 'READY_FOR_T028'
assert candidate['exact_revision'] == '6f9c4b12a95b139af68670a6713616b757923735'
assert candidate['authentication_required'] is False
assert candidate['gated_access'] is False
assert candidate['expected_monetary_cost'] == 'USD 0.00'
print(candidate['exact_model_id'], candidate['exact_revision'])


In [ ]:
REPORT = OUT / 'q4-ministral-3-3b.json'
WORKDIR = Path('/content/mstr_t029_ministral')
cmd = [
    'python', str(RUNNER),
    '--t027-manifest', str(MANIFEST),
    '--candidate', CANDIDATE,
    '--llama-cpp-commit', LLAMA_CPP_COMMIT,
    '--workdir', str(WORKDIR),
    '--report', str(REPORT),
]
completed = subprocess.run(cmd, cwd=REPO)
assert REPORT.is_file(), 'Runner did not emit a durable report'
report = json.loads(REPORT.read_text(encoding='utf-8'))
print(json.dumps(report, indent=2)[:12000])
assert completed.returncode == 0, f"runner rc={completed.returncode}; classification={report.get('result_classification')}"
assert report['candidate_id'] == CANDIDATE
assert report['model_revision'] == candidate['exact_revision']
assert report['tool']['exact_commit'] == LLAMA_CPP_COMMIT
assert report['result_classification'] in {'Q4_PROFILE_READY', 'Q4_PROFILE_PARTIAL'}


In [ ]:
import hashlib, shutil
payload = REPORT.read_bytes()
print('report_path =', REPORT)
print('report_sha256 =', hashlib.sha256(payload).hexdigest())
print('report_bytes =', len(payload))
shutil.rmtree(WORKDIR, ignore_errors=True)
print('ephemeral binary workdir removed =', not WORKDIR.exists())


Download only `/content/reports/q4-ministral-3-3b.json` and attach/commit it through the governed T029 evidence flow. Do not persist GGUF, safetensors, build trees, or model binaries outside the ephemeral VM.
